In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

In [ ]:
# --- WaveDrom / VCD rendering utilities (auto-generated — do not edit) ---
import json, re, subprocess
from IPython.display import HTML, display

def _vcd_to_wavedrom(vcd_path, max_cycles=80, signals=None):
    """
    Minimal VCD → WaveDrom JSON converter.
    Works for single-bit and multi-bit signals from iverilog output.
    """
    with open(vcd_path) as f:
        vcd = f.read()

    # Parse variable declarations
    var_map = {}  # id_code → (name, width)
    for m in re.finditer(
        r"\$var\s+\w+\s+(\d+)\s+(\S+)\s+(\S+)", vcd
    ):
        width, code, name = int(m.group(1)), m.group(2), m.group(3)
        if signals is None or name in signals:
            var_map[code] = (name, width)

    if not var_map:
        print(f"⚠  No matching signals in {vcd_path}")
        return None

    # Parse value changes
    changes = {}  # id_code → [(time, value), ...]
    current_time = 0
    for line in vcd.splitlines():
        line = line.strip()
        if line.startswith("#"):
            current_time = int(line[1:])
        elif line.startswith("b"):
            parts = line.split()
            if len(parts) == 2 and parts[1] in var_map:
                val = int(parts[0][1:], 2) if parts[0][1:] else 0
                changes.setdefault(parts[1], []).append((current_time, val))
        elif len(line) >= 2 and line[0] in "01xXzZ" and line[1:] in var_map:
            code = line[1:]
            val = {"0": 0, "1": 1, "x": "x", "X": "x", "z": "z", "Z": "z"}[line[0]]
            changes.setdefault(code, []).append((current_time, val))

    # Determine time step (smallest non-zero delta)
    all_times = sorted({t for ch in changes.values() for t, _ in ch})
    if len(all_times) < 2:
        return None
    deltas = [all_times[i+1] - all_times[i] for i in range(len(all_times)-1) if all_times[i+1] != all_times[i]]
    if not deltas:
        return None
    step = min(deltas)
    end_time = min(all_times[-1], step * max_cycles) if max_cycles else all_times[-1]
    sample_times = list(range(0, end_time + 1, step))[:max_cycles]

    # Build WaveDrom signal list
    wave_signals = []
    for code, (name, width) in sorted(var_map.items(), key=lambda x: x[1][0]):
        ch = sorted(changes.get(code, []))
        # Sample at each time point
        samples = []
        cur_val = 0
        ci = 0
        for t in sample_times:
            while ci < len(ch) and ch[ci][0] <= t:
                cur_val = ch[ci][1]
                ci += 1
            samples.append(cur_val)

        if width == 1:
            # Single-bit: WaveDrom wave string
            wave_str = ""
            for v in samples:
                if v == "x":
                    wave_str += "x"
                elif v == "z":
                    wave_str += "z"
                else:
                    wave_str += str(int(v))
            wave_signals.append({"name": name, "wave": wave_str})
        else:
            # Multi-bit: use '=' for data with labels
            wave_str = ""
            data = []
            prev = None
            for v in samples:
                if v == prev:
                    wave_str += "."
                else:
                    wave_str += "="
                    data.append(f"0x{v:X}" if isinstance(v, int) else str(v))
                    prev = v
            wave_signals.append({"name": name, "wave": wave_str, "data": data})

    return {"signal": wave_signals, "config": {"hscale": 1}}


def show_waves(vcd_path="dump.vcd", max_cycles=80, signals=None, width=900):
    """Render VCD waveforms inline using WaveDrom."""
    wd = _vcd_to_wavedrom(vcd_path, max_cycles=max_cycles, signals=signals)
    if wd is None:
        print("No waveform data to display.")
        return
    wd_json = json.dumps(wd)
    html = f"""
    <div id="wd_{id(wd):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wd):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wd):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))


def show_wavedrom(wave_dict, width=900):
    """Render a raw WaveDrom JSON dict inline."""
    wd_json = json.dumps(wave_dict)
    html = f"""
    <div id="wd_{id(wave_dict):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wave_dict):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wave_dict):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))

print("✓ WaveDrom helpers loaded — use show_waves('dump.vcd') after simulation")

# Day 12 Lab: UART RX, SPI & IP Integration

## Overview
Build the UART receiver with 16× oversampling, create a loopback test, and
explore SPI master design or IP integration.

## Prerequisites
- Working UART TX from Day 11
- Pre-class video on UART RX oversampling and SPI

## Exercises

| # | Exercise | Time | Key SLOs |
|---|----------|------|----------|
| 1 | UART RX Module + Testbench | 40 min | 12.1, 12.2 |
| 2 | UART Loopback on Hardware | 25 min | 12.3 |
| 3 | SPI Master Module | 30 min | 12.4, 12.5 |
| 4 | UART-Controlled LED Pattern | 15 min | 12.3 |
| 5 | UART-to-SPI Bridge (Stretch) | 15 min | 12.4, 12.5 |

## Deliverables
1. UART RX with all test bytes passing in simulation
2. **Loopback working on hardware** — type on PC, see echo (crown jewel!)
3. SPI master verified in simulation

---
## Exercise Files

The starter files for each exercise are below. Edit the code, then run the simulation/build cells.

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile uart_rx.v
// =============================================================================
// uart_rx.v — UART Receiver with 16× Oversampling (8N1)
// Day 12, Exercise 1
// =============================================================================

module uart_rx #(
    parameter CLK_FREQ  = 25_000_000,
    parameter BAUD_RATE = 115_200
)(
    input  wire       i_clk,
    input  wire       i_reset,
    input  wire       i_rx,        // serial input line
    output reg  [7:0] o_data,      // received byte
    output reg        o_valid      // pulsed high for 1 cycle when byte ready
);

    localparam CLKS_PER_BIT    = CLK_FREQ / BAUD_RATE;
    localparam CLKS_PER_SAMPLE = CLKS_PER_BIT / 16;     // 16× oversample

    localparam S_IDLE    = 3'd0;
    localparam S_START   = 3'd1;
    localparam S_DATA    = 3'd2;
    localparam S_STOP    = 3'd3;
    localparam S_CLEANUP = 3'd4;

    // 2-FF synchronizer for async i_rx
    reg r_rx_sync1, r_rx_sync2;
    always @(posedge i_clk) begin
        r_rx_sync1 <= i_rx;
        r_rx_sync2 <= r_rx_sync1;
    end

    reg [2:0]  r_state;
    reg [$clog2(CLKS_PER_SAMPLE)-1:0] r_sample_cnt;
    reg [3:0]  r_oversample_cnt;   // 0–15 per bit
    reg [2:0]  r_bit_idx;
    reg [7:0]  r_shift;

    wire w_sample_tick = (r_sample_cnt == CLKS_PER_SAMPLE - 1);

    // TODO: Implement the RX FSM
    always @(posedge i_clk) begin
        if (i_reset) begin
            r_state         <= S_IDLE;
            r_sample_cnt    <= 0;
            r_oversample_cnt <= 0;
            r_bit_idx       <= 0;
            r_shift         <= 0;
            o_data          <= 0;
            o_valid         <= 0;
        end else begin
            o_valid <= 1'b0;

            case (r_state)
                S_IDLE: begin
                    r_sample_cnt     <= 0;
                    r_oversample_cnt <= 0;
                    r_bit_idx        <= 0;
                    // TODO: Detect falling edge on r_rx_sync2
                    //       (line goes from high to low = start bit)
                    //       Transition to S_START

                    // ---- YOUR CODE HERE ----
                end

                S_START: begin
                    // TODO: Count oversample ticks
                    //       At tick 7 (center of start bit):
                    //         if r_rx_sync2 is still low → valid start bit → S_DATA
                    //         if r_rx_sync2 is high → glitch → S_IDLE

                    // ---- YOUR CODE HERE ----
                end

                S_DATA: begin
                    // TODO: Count oversample ticks (16 per bit)
                    //       At tick 15 (end of bit period), sample r_rx_sync2
                    //       Shift into r_shift: r_shift <= {r_rx_sync2, r_shift[7:1]}
                    //       After 8 bits (r_bit_idx == 7), go to S_STOP

                    // ---- YOUR CODE HERE ----
                end

                S_STOP: begin
                    // TODO: Count 16 oversample ticks for stop bit
                    //       At tick 15: output r_shift to o_data,
                    //       pulse o_valid, go to S_CLEANUP

                    // ---- YOUR CODE HERE ----
                end

                S_CLEANUP: begin
                    r_state <= S_IDLE;
                end
            endcase
        end
    end

endmodule

In [ ]:
%%writefile Makefile
# Makefile — uart_rx
PROJECT  = uart_rx
TOP      = uart_rx
PCF      = ../go_board.pcf
SRCS     = uart_rx.v uart_tx.v
TB       = tb_uart_rx.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile top_uart_loopback.v
// =============================================================================
// top_uart_loopback.v — UART Loopback (RX → TX echo)
// Day 12, Exercise 2
// =============================================================================
// Type on PC terminal → echoes back. Crown jewel of Week 3.

module top_uart_loopback (
    input  wire i_clk,
    input  wire i_uart_rx,
    output wire o_uart_tx,
    output wire o_segment1_a, o_segment1_b, o_segment1_c,
    output wire o_segment1_d, o_segment1_e, o_segment1_f, o_segment1_g,
    output wire o_led1, o_led2, o_led3, o_led4
);

    wire [7:0] w_rx_data;
    wire       w_rx_valid;

    uart_rx #(.CLK_FREQ(25_000_000), .BAUD_RATE(115_200)) rx (
        .i_clk(i_clk), .i_reset(1'b0),
        .i_rx(i_uart_rx),
        .o_data(w_rx_data), .o_valid(w_rx_valid)
    );

    // TODO: Connect RX output to TX input for echo
    //   When rx asserts o_valid, feed w_rx_data into the TX

    uart_tx #(.CLK_FREQ(25_000_000), .BAUD_RATE(115_200)) tx (
        .i_clk(i_clk), .i_reset(1'b0),
        // TODO: Connect i_valid and i_data
        .i_valid(/* ---- YOUR CODE ---- */),
        .i_data(/* ---- YOUR CODE ---- */),
        .o_tx(o_uart_tx), .o_busy()
    );

    // TODO: Display last received byte on 7-segment
    //   Latch w_rx_data when w_rx_valid is high
    //   Show lower nibble on segment1

    // ---- YOUR CODE HERE ----

    // LED activity indicator
    reg r_activity;
    always @(posedge i_clk) begin
        if (w_rx_valid) r_activity <= ~r_activity;
    end
    assign o_led1 = ~r_activity;
    assign o_led2 = 1'b1;
    assign o_led3 = 1'b1;
    assign o_led4 = 1'b1;

endmodule

In [ ]:
%%writefile Makefile
# Makefile — top_uart_loopback
PROJECT  = top_uart_loopback
TOP      = top_uart_loopback
PCF      = ../go_board.pcf
SRCS     = top_uart_loopback.v uart_tx.v uart_rx.v hex_to_7seg.v
TB       = tb_top_uart_loopback.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: $(PROJECT).bin

$(PROJECT).json: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $@" $(SRCS)

$(PROJECT).asc: $(PROJECT).json $(PCF)
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $< --asc $@

$(PROJECT).bin: $(PROJECT).asc
	icepack $< $@

prog: $(PROJECT).bin
	iceprog $<

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile spi_master.v
// =============================================================================
// spi_master.v — SPI Master Module (Mode 0)
// Day 12, Exercise 3
// =============================================================================

module spi_master #(
    parameter CLK_FREQ  = 25_000_000,
    parameter SPI_FREQ  = 1_000_000,
    parameter DATA_BITS = 8
)(
    input  wire                  i_clk,
    input  wire                  i_reset,
    input  wire                  i_start,
    input  wire [DATA_BITS-1:0]  i_tx_data,
    output reg  [DATA_BITS-1:0]  o_rx_data,
    output reg                   o_done,
    output wire                  o_busy,
    output reg                   o_sclk,
    output reg                   o_mosi,
    input  wire                  i_miso,
    output reg                   o_cs_n
);

    localparam CLKS_PER_HALF_SCLK = CLK_FREQ / (SPI_FREQ * 2);

    localparam S_IDLE    = 2'b00;
    localparam S_RUNNING = 2'b01;
    localparam S_DONE    = 2'b10;

    reg [1:0] r_state;
    reg [$clog2(CLKS_PER_HALF_SCLK)-1:0] r_sclk_count;
    reg [$clog2(DATA_BITS)-1:0] r_bit_count;
    reg [DATA_BITS-1:0] r_tx_shift, r_rx_shift;
    reg r_sclk_edge;

    assign o_busy = (r_state != S_IDLE);

    // TODO: Implement the SPI master FSM (Mode 0: CPOL=0, CPHA=0)
    //   IDLE: CS high, SCLK low. On i_start: assert CS, load shift reg
    //   RUNNING: Toggle SCLK at SPI_FREQ rate
    //     - Falling SCLK: shift out MOSI (MSB first)
    //     - Rising SCLK: sample MISO into shift reg
    //     - After DATA_BITS clocks: go to DONE
    //   DONE: Deassert CS, pulse o_done, return to IDLE

    // ---- YOUR CODE HERE ----

endmodule

In [ ]:
%%writefile Makefile
# Makefile — spi_master
PROJECT  = spi_master
TOP      = spi_master
PCF      = ../go_board.pcf
SRCS     = $(wildcard *.v)
TB       = tb_spi_master.v

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean